# ERA5 (WeatherBench2) vs ASOS 비교 분석
## 2020년 서울 지역 — 재분석 자료 vs 지상 관측 자료

---

### 데이터 출처

| 구분 | 데이터 | 해상도 | 시간간격 | 좌표 |
|------|--------|--------|----------|------|
| ERA5 | WeatherBench2 GCS Zarr | 1.5° × 1.5° | 6시간 (UTC) | 37.5°N, 127.5°E |
| ASOS | 기상청 종관기상관측 | 지점 관측 | 1시간 (KST) | 서울 (STN 108) |

### 비교 변수

| ERA5 변수 | ASOS 변수 | 설명 | 단위 |
|-----------|-----------|------|------|
| `2m_temperature` | `TA` | 기온 | °C |
| `total_precipitation_6hr` | `RN` (6h 합산) | 강수량 | mm |
| `2m_dewpoint_temperature` | `TD` | 이슬점온도 | °C |
| `mean_sea_level_pressure` | `PS` | 해면기압 | hPa |
| `10m_wind_speed` | `WS` | 풍속 | m/s |

### 시간 처리
- ASOS KST → UTC 변환 후 6시간 리샘플링
- ERA5 6시간 타임스텝과 매칭

---
## Step 1. 패키지 임포트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/weather-climate-ai-tutorials

In [ ]:
# 나눔 고딕 폰트 설치
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import gcsfs
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pathlib, urllib.request, matplotlib, matplotlib.font_manager as fm


# Install zarr and fsspec if not already installed
!pip install zarr fsspec

# ── seaborn 스타일 먼저 적용 (이후 rcParams 덮어쓰기 방지) ──────────
sns.set_style('whitegrid')

# ── 한글 폰트 설치 및 등록 ───────────────────────────────────────────
font_path = pathlib.Path(matplotlib.get_data_path()) / 'fonts' / 'ttf' / 'NanumGothic.ttf'

if not font_path.exists():
    print('NanumGothic 폰트 다운로드 중...')
    try:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf',
            font_path
        )
        print('다운로드 완료')
    except Exception as e:
        print(f'다운로드 실패: {e}')

if font_path.exists():
    fm.fontManager.addfont(str(font_path))
    _font_name = fm.FontProperties(fname=str(font_path)).get_name()
    plt.rcParams['font.family'] = _font_name
    print(f'폰트 등록 완료: {_font_name}')
else:
    for sys_path in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf']:
        if pathlib.Path(sys_path).exists():
            fm.fontManager.addfont(sys_path)
            plt.rcParams['font.family'] = fm.FontProperties(fname=sys_path).get_name()
            print(f'시스템 폰트 사용: {sys_path}')
            break
    else:
        print('한글 폰트 없음 → 기본 폰트 사용')

plt.rcParams['axes.unicode_minus'] = False

print(f'xarray  : {xr.__version__}')
print(f'pandas  : {pd.__version__}')
print(f'폰트    : {plt.rcParams["font.family"]}')

---
## Step 2. ERA5 데이터 로드 (WeatherBench2 GCS)

In [ ]:
# ── GCS 연결 ─────────────────────────────────────────────────────────
SEOUL_LAT = 37.56
SEOUL_LON = 126.97
YEAR      = 2020

fs = gcsfs.GCSFileSystem(token='anon')
GCS_PATH = (
    'gs://weatherbench2/datasets/era5/'
    '1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr'
)
store = fs.get_mapper(GCS_PATH)
ds = xr.open_zarr(store, consolidated=True, chunks={'time': 100})

# ── 서울 격자점 + 2020년 필터링 ──────────────────────────────────────
ERA5_VARS = [
    '2m_temperature',
    'total_precipitation_6hr',
    'mean_sea_level_pressure',
    '10m_wind_speed',
]

ds_seoul = (
    ds[ERA5_VARS]
    .sel(time=slice(f'{YEAR}-01-01', f'{YEAR}-12-31T23:59:59'))
    .sel(latitude=SEOUL_LAT, longitude=SEOUL_LON, method='nearest')
)

# ── 상대습도: pressure level 변수 → 1000 hPa 선택 ────────────────────
ds_rh = (
    ds['relative_humidity']
    .sel(level=1000)
    .sel(time=slice(f'{YEAR}-01-01', f'{YEAR}-12-31T23:59:59'))
    .sel(latitude=SEOUL_LAT, longitude=SEOUL_LON, method='nearest')
)

sel_lat = float(ds_seoul.latitude)
sel_lon = float(ds_seoul.longitude)
print(f'요청 좌표 : {SEOUL_LAT}°N, {SEOUL_LON}°E')
print(f'선택 격자 : {sel_lat:.2f}°N, {sel_lon:.2f}°E')
print(f'타임스텝  : {len(ds_seoul.time)}개 (6시간 간격)')

In [ ]:
# ── Dask 계산 실행 + DataFrame 변환 ──────────────────────────────────
print('ERA5 데이터 다운로드 중...')
ds_seoul = ds_seoul.compute()
ds_rh    = ds_rh.compute()

df_era5 = ds_seoul.to_dataframe().reset_index()[['time'] + ERA5_VARS].copy()
df_era5 = df_era5.rename(columns={
    '2m_temperature'         : 'era5_T2m',
    'total_precipitation_6hr': 'era5_RN',
    'mean_sea_level_pressure': 'era5_PS',
    '10m_wind_speed'         : 'era5_WS',
})

# 단위 변환
df_era5['era5_T2m'] -= 273.15        # K → °C
df_era5['era5_RN']  *= 1000          # m → mm
df_era5['era5_PS']  /= 100           # Pa → hPa
df_era5['era5_RN']  = df_era5['era5_RN'].clip(lower=0)  # 음수 제거

# 상대습도 병합 (1000 hPa pressure level, fraction → %)
df_rh = ds_rh.to_dataframe().reset_index()[['time', 'relative_humidity']].copy()
df_rh['relative_humidity'] *= 100    # 0~1 → 0~100 %
df_rh = df_rh.rename(columns={'relative_humidity': 'era5_RH'})
df_era5 = df_era5.merge(df_rh, on='time', how='left')

# time을 UTC timezone-aware로
df_era5['time'] = pd.to_datetime(df_era5['time']).dt.tz_localize('UTC')
df_era5 = df_era5.set_index('time').sort_index()

print(f'ERA5 로드 완료: {df_era5.shape}')
print(f'※ 상대습도: 1000 hPa pressure level, fraction → % 변환 완료')
print(f'era5_RH 범위: {df_era5["era5_RH"].min():.1f} ~ {df_era5["era5_RH"].max():.1f} %')
df_era5.head()

---
## Step 3. ASOS 데이터 로드 및 전처리

In [ ]:
# ── ASOS merged CSV 로드 ──────────────────────────────────────────────
ASOS_PATH = './asos_data/asos_2020_merged.csv'

df_asos_raw = pd.read_csv(ASOS_PATH, dtype=str, low_memory=False)

# datetime 파싱 (KST)
df_asos_raw['datetime_kst'] = pd.to_datetime(
    df_asos_raw['TM'], format='%Y%m%d%H%M', errors='coerce'
)

# 수치형 변환
for col in ['TA', 'TD', 'PS', 'WS', 'RN', 'HM', 'PV', 'SS']:
    if col in df_asos_raw.columns:
        df_asos_raw[col] = pd.to_numeric(df_asos_raw[col], errors='coerce')
        df_asos_raw.loc[df_asos_raw[col] <= -9, col] = np.nan

df_asos_raw = df_asos_raw.dropna(subset=['datetime_kst']).copy()

# KST → UTC 변환
df_asos_raw['datetime_utc'] = (
    df_asos_raw['datetime_kst']
    .dt.tz_localize('Asia/Seoul')
    .dt.tz_convert('UTC')
)
df_asos_raw = df_asos_raw.set_index('datetime_utc').sort_index()

print(f'ASOS 로드 완료: {df_asos_raw.shape}')
print(f'기간 (UTC): {df_asos_raw.index.min()} ~ {df_asos_raw.index.max()}')
df_asos_raw[['TA','TD','PS','WS','RN','HM']].head()

In [ ]:
# ── ASOS 1시간 → 6시간 리샘플링 ──────────────────────────────────────
# ERA5 타임스텝(00, 06, 12, 18 UTC)에 맞춰 집계
# - 기온/이슬점/기압/풍속 : 6시간 평균
# - 강수량 : 6시간 합산

df_asos_6h = df_asos_raw[['TA','TD','PS','WS','RN','HM']].resample('6h').agg({
    'TA': 'mean',
    'TD': 'mean',
    'PS': 'mean',
    'WS': 'mean',
    'RN': 'sum',
    'HM': 'mean',
})

df_asos_6h = df_asos_6h.rename(columns={
    'TA': 'asos_T2m',
    'TD': 'asos_TD',
    'PS': 'asos_PS',
    'WS': 'asos_WS',
    'RN': 'asos_RN',
    'HM': 'asos_HM',
})

print(f'ASOS 6h 리샘플링 완료: {df_asos_6h.shape}')
df_asos_6h.head()

---
## Step 4. 데이터 병합 및 검증

In [ ]:
# ── inner join (공통 타임스텝만 유지) ────────────────────────────────
df = df_era5.join(df_asos_6h, how='inner')
df.index.name = 'time_utc'

print(f'병합 완료: {df.shape}  ({len(df)}개 공통 타임스텝)')
print(f'기간: {df.index.min()} ~ {df.index.max()}')
print(f'\n결측률:')
print((df.isna().mean() * 100).round(2).to_string())
df.head(10)

In [ ]:
# ── 변수별 Bias / RMSE / 상관계수 ────────────────────────────────────
PAIRS = [
    ('era5_T2m', 'asos_T2m', '기온 (°C)'),
    ('era5_RN',  'asos_RN',  '6h 강수량 (mm)'),
    ('era5_PS',  'asos_PS',  '해면기압 (hPa)'),
    ('era5_WS',  'asos_WS',  '풍속 (m/s)'),
    ('era5_RH',  'asos_HM',  '상대습도 (%) [ERA5: 1000hPa]'),
]

print(f"{'변수':<32} {'Bias':>8} {'MAE':>8} {'RMSE':>8} {'r':>7} {'n':>6}")
print('-' * 74)

metrics = {}
for ecol, acol, label in PAIRS:
    sub = df[[ecol, acol]].dropna()
    if len(sub) == 0:
        print(f'{label:<32} {"데이터 없음":>8}')
        continue
    e, a  = sub[ecol], sub[acol]
    bias  = float((e - a).mean())
    mae   = float((e - a).abs().mean())
    rmse  = float(np.sqrt(((e - a)**2).mean()))
    r     = float(e.corr(a))
    metrics[label] = {'Bias': bias, 'MAE': mae, 'RMSE': rmse, 'r': r, 'n': len(sub)}
    print(f'{label:<32} {bias:>8.3f} {mae:>8.3f} {rmse:>8.3f} {r:>7.3f} {len(sub):>6}')

print()
print('※ 상대습도: ERA5 1000 hPa pressure level vs ASOS 지표 관측 (HM)')

---
## Step 5. 시계열 비교

In [ ]:
# ── 주요 변수 시계열 비교 ─────────────────────────────────────────────
PLOT_PAIRS = [
    ('era5_T2m', 'asos_T2m', '기온 (°C)',                        'tomato',  'steelblue'),
    ('era5_PS',  'asos_PS',  '해면기압 (hPa)',                   'purple',  'darkcyan'),
    ('era5_WS',  'asos_WS',  '풍속 (m/s)',                       'crimson', 'navy'),
    ('era5_RH',  'asos_HM',  '상대습도 (%) [ERA5: 1000 hPa]',   'teal',    'sienna'),
]

fig, axes = plt.subplots(len(PLOT_PAIRS), 1, figsize=(18, 4 * len(PLOT_PAIRS)), sharex=True)

for ax, (ecol, acol, label, ec, ac) in zip(axes, PLOT_PAIRS):
    ax.plot(df.index, df[ecol], color=ec, lw=0.8, alpha=0.85, label='ERA5')
    ax.plot(df.index, df[acol], color=ac, lw=0.8, alpha=0.85, label='ASOS')
    ax.set_ylabel(label, fontsize=10)
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)

    m = metrics.get(label, {})
    if m:
        ax.set_title(
            f'{label}   Bias={m["Bias"]:+.2f}  RMSE={m["RMSE"]:.2f}  r={m["r"]:.3f}',
            fontsize=11
        )

axes[-1].set_xlabel('날짜 (UTC)', fontsize=10)
plt.suptitle(
    f'ERA5 vs ASOS 시계열 비교 — 2020년 서울\n'
    f'ERA5 격자: {sel_lat:.1f}°N {sel_lon:.1f}°E  /  ASOS STN 108',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('era5_asos_timeseries_2020.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 강수량 시계열 비교 (일별 합산 바 차트) ────────────────────────────
# 6h → 일별 합산 (막대가 너무 얇아지는 문제 방지)
daily_era5 = df['era5_RN'].resample('1D').sum()
daily_asos = df['asos_RN'].resample('1D').sum()

fig, axes = plt.subplots(2, 1, figsize=(18, 7), sharex=True)

axes[0].bar(daily_era5.index, daily_era5, width=0.8, color='steelblue',
            alpha=0.8, label='ERA5')
axes[0].set_ylabel('일 강수량 (mm)')
axes[0].set_title('ERA5 일별 강수량 (6h 합산)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].bar(daily_asos.index, daily_asos, width=0.8, color='tomato',
            alpha=0.8, label='ASOS')
axes[1].set_ylabel('일 강수량 (mm)')
axes[1].set_title('ASOS 일별 강수량 (1h 합산)', fontsize=11)
axes[1].set_xlabel('날짜 (UTC)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.suptitle('ERA5 vs ASOS 일별 강수량 비교 — 2020년 서울', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 6. 월별 비교

In [ ]:
# ── 월별 평균/합계 비교 ───────────────────────────────────────────────
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
df['month'] = df.index.month

# 기온 월별 평균
monthly_T = df.groupby('month')[['era5_T2m','asos_T2m']].mean()
# 강수량 월별 합계
monthly_RN = df.groupby('month')[['era5_RN','asos_RN']].sum()
# 기압 월별 평균
monthly_PS = df.groupby('month')[['era5_PS','asos_PS']].mean()
# 풍속 월별 평균
monthly_WS = df.groupby('month')[['era5_WS','asos_WS']].mean()

x = np.arange(12)
w = 0.35

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 기온
axes[0,0].plot(x, monthly_T['era5_T2m'], 'o-', color='tomato',    lw=2, ms=6, label='ERA5')
axes[0,0].plot(x, monthly_T['asos_T2m'], 's-', color='steelblue', lw=2, ms=6, label='ASOS')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(month_names, rotation=45, fontsize=9)
axes[0,0].set_ylabel('평균 기온 (°C)'); axes[0,0].set_title('월별 평균 기온', fontsize=12)
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# 강수량
axes[0,1].bar(x - w/2, monthly_RN['era5_RN'], width=w, color='steelblue', alpha=0.8, label='ERA5')
axes[0,1].bar(x + w/2, monthly_RN['asos_RN'], width=w, color='tomato',    alpha=0.8, label='ASOS')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(month_names, rotation=45, fontsize=9)
axes[0,1].set_ylabel('총 강수량 (mm)'); axes[0,1].set_title('월별 총 강수량', fontsize=12)
axes[0,1].legend(); axes[0,1].grid(axis='y', alpha=0.3)

# 기압
axes[1,0].plot(x, monthly_PS['era5_PS'], 'o-', color='purple',  lw=2, ms=6, label='ERA5')
axes[1,0].plot(x, monthly_PS['asos_PS'], 's-', color='darkcyan',lw=2, ms=6, label='ASOS')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(month_names, rotation=45, fontsize=9)
axes[1,0].set_ylabel('해면기압 (hPa)'); axes[1,0].set_title('월별 평균 해면기압', fontsize=12)
axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# 풍속
axes[1,1].plot(x, monthly_WS['era5_WS'], 'o-', color='crimson', lw=2, ms=6, label='ERA5')
axes[1,1].plot(x, monthly_WS['asos_WS'], 's-', color='navy',    lw=2, ms=6, label='ASOS')
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(month_names, rotation=45, fontsize=9)
axes[1,1].set_ylabel('평균 풍속 (m/s)'); axes[1,1].set_title('월별 평균 풍속', fontsize=12)
axes[1,1].legend(); axes[1,1].grid(alpha=0.3)

plt.suptitle('ERA5 vs ASOS 월별 비교 — 2020년 서울', fontsize=14)
plt.tight_layout()
plt.savefig('era5_asos_monthly_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 7. 산점도 (ERA5 vs ASOS)

In [ ]:
# ── 변수별 산점도 ─────────────────────────────────────────────────────
SCATTER_PAIRS = [
    ('era5_T2m', 'asos_T2m', '기온 (°C)',                       '#E63946'),
    ('era5_PS',  'asos_PS',  '해면기압 (hPa)',                  '#9C27B0'),
    ('era5_WS',  'asos_WS',  '풍속 (m/s)',                      '#2196F3'),
    ('era5_RH',  'asos_HM',  '상대습도 (%) [ERA5: 1000 hPa]',  '#009688'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes_flat = axes.ravel()

for ax, (ecol, acol, label, color) in zip(axes_flat, SCATTER_PAIRS):
    sub = df[[ecol, acol]].dropna()
    if len(sub) == 0:
        ax.set_visible(False)
        continue

    ax.scatter(sub[acol], sub[ecol], s=3, alpha=0.3, color=color)

    vmin = min(sub[ecol].min(), sub[acol].min())
    vmax = max(sub[ecol].max(), sub[acol].max())
    ax.plot([vmin, vmax], [vmin, vmax], 'k--', lw=1.2, label='1:1 선')

    z = np.polyfit(sub[acol], sub[ecol], 1)
    p = np.poly1d(z)
    xs = np.linspace(vmin, vmax, 100)
    ax.plot(xs, p(xs), 'r-', lw=1.5, alpha=0.8, label=f'회귀선 (slope={z[0]:.2f})')

    m = metrics.get(label, {})
    info = f"Bias={m.get('Bias',0):+.2f}  RMSE={m.get('RMSE',0):.2f}  r={m.get('r',0):.3f}" if m else ''
    ax.set_xlabel(f'ASOS {label}', fontsize=10)
    ax.set_ylabel(f'ERA5 {label}', fontsize=10)
    ax.set_title(f'{label}\n{info}', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('ERA5 vs ASOS 산점도 — 2020년 서울 (6시간 데이터)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('era5_asos_scatter_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8. 계절별 Bias 분석

In [ ]:
# ── 계절별 ERA5 - ASOS 편차 ───────────────────────────────────────────
season_map = {
    12:'겨울',1:'겨울',2:'겨울',
    3:'봄',  4:'봄',  5:'봄',
    6:'여름',7:'여름',8:'여름',
    9:'가을',10:'가을',11:'가을'
}
df['season'] = df['month'].map(season_map)

df['bias_T2m'] = df['era5_T2m'] - df['asos_T2m']
df['bias_PS']  = df['era5_PS']  - df['asos_PS']
df['bias_WS']  = df['era5_WS']  - df['asos_WS']
df['bias_RH']  = df['era5_RH']  - df['asos_HM']

BIAS_COLS = [
    ('bias_T2m', '기온 Bias (°C)',                     'tomato'),
    ('bias_PS',  '해면기압 Bias (hPa)',                'purple'),
    ('bias_WS',  '풍속 Bias (m/s)',                    'steelblue'),
    ('bias_RH',  '상대습도 Bias (%) [ERA5: 1000 hPa]', 'teal'),
]

season_order  = ['봄', '여름', '가을', '겨울']
season_colors = {'봄':'#4CAF50', '여름':'#E63946', '가을':'#FF9800', '겨울':'#2196F3'}

df_valid = df[df['season'].notna()]

# ── 계절별 통계 테이블 출력 ───────────────────────────────────────────
print(f"{'':30}", end='')
for s in season_order:
    print(f"  {s:^22}", end='')
print()
print(f"{'변수':<30}", end='')
for s in season_order:
    print(f"  {'Mean':>6} {'Median':>6} {'IQR':>6}", end='')
print()
print('-' * (30 + 24 * len(season_order)))

for bcol, label, _ in BIAS_COLS:
    print(f'{label:<30}', end='')
    for s in season_order:
        vals = df_valid.loc[df_valid['season'] == s, bcol].dropna()
        if len(vals) == 0:
            print(f"  {'N/A':>6} {'N/A':>6} {'N/A':>6}", end='')
        else:
            q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
            print(f"  {vals.mean():>+6.2f} {vals.median():>+6.2f} {q3-q1:>6.2f}", end='')
    print()

print()
print('※ 상대습도 Bias: ERA5 1000 hPa pressure level − ASOS 지표 관측')

# ── 박스플롯 ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes_flat = axes.ravel()

for ax, (bcol, label, _) in zip(axes_flat, BIAS_COLS):
    data_by_season = [df_valid.loc[df_valid['season'] == s, bcol].dropna().values
                      for s in season_order]

    non_empty = [v for v in data_by_season if len(v) > 0]
    if not non_empty:
        ax.set_visible(False)
        continue

    bp = ax.boxplot(
        data_by_season,
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(linewidth=1.2),
        capprops=dict(linewidth=1.2),
        flierprops=dict(marker='o', markersize=3, alpha=0.4),
    )
    for patch, s in zip(bp['boxes'], season_order):
        patch.set_facecolor(season_colors[s])
        patch.set_alpha(0.75)

    # ── 수치 어노테이션 ───────────────────────────────────────────────
    all_vals = np.concatenate(non_empty)
    y_min, y_max = all_vals.min(), all_vals.max()
    y_range = y_max - y_min if y_max != y_min else 1.0
    text_y = y_min - y_range * 0.04

    for i, (vals, s) in enumerate(zip(data_by_season, season_order), start=1):
        if len(vals) == 0:
            continue
        q1, q3 = np.percentile(vals, 25), np.percentile(vals, 75)
        txt = (f"mean={vals.mean():+.2f}\n"
               f"med={np.median(vals):+.2f}\n"
               f"IQR={q3-q1:.2f}")
        ax.text(i, text_y, txt,
                ha='center', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7, ec='gray', lw=0.5))

    ax.set_ylim(text_y - y_range * 0.32, y_max + y_range * 0.08)
    ax.set_xticks(range(1, len(season_order) + 1))
    ax.set_xticklabels(season_order, fontsize=11)
    ax.axhline(0, color='black', linestyle='--', lw=1.5)
    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=10)
    ax.set_title(f'계절별 {label}\n(ERA5 − ASOS)', fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('ERA5 vs ASOS 계절별 Bias (ERA5 − ASOS) — 2020년 서울\n'
             '※ 상대습도: ERA5 1000 hPa pressure level vs ASOS 지표 관측', fontsize=13)
plt.tight_layout()
plt.savefig('era5_asos_seasonal_bias_2020.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9. 종합 요약

In [ ]:
# ── 종합 통계 요약 ────────────────────────────────────────────────────
print('=' * 65)
print('ERA5 vs ASOS 비교 요약 — 2020년 서울')
print(f'ERA5 격자: {sel_lat:.1f}°N, {sel_lon:.1f}°E  /  ASOS STN 108')
print('=' * 65)

summary_rows = []
for ecol, acol, label in PAIRS:
    sub = df[[ecol, acol]].dropna()
    if len(sub) == 0:
        continue
    e, a = sub[ecol], sub[acol]
    summary_rows.append({
        '변수'     : label,
        'ERA5 평균': round(e.mean(), 3),
        'ASOS 평균': round(a.mean(), 3),
        'Bias'     : round((e - a).mean(), 3),
        'MAE'      : round((e - a).abs().mean(), 3),
        'RMSE'     : round(np.sqrt(((e - a)**2).mean()), 3),
        'r'        : round(e.corr(a), 4),
        'n'        : len(sub),
    })

summary_df = pd.DataFrame(summary_rows).set_index('변수')
print(summary_df.to_string())

print()
print('주요 해석 포인트:')
notes = [
    'ERA5는 1.5° 격자 재분석 자료로 서울 도심 관측(ASOS)과 공간 대표성 차이가 존재',
    '기온 Bias가 양수(+)이면 ERA5가 ASOS보다 고온 → 도시 열섬 효과 미반영 가능성',
    '강수량은 ERA5가 공간 평균값이므로 국지 집중호우를 과소평가하는 경향',
    '풍속은 지형/건물 영향으로 ASOS가 ERA5보다 낮게 관측되는 경향',
    '해면기압은 두 자료 간 상관이 높아(r > 0.99) 신뢰성 검증 기준으로 활용 가능',
]
for note in notes:
    print(f'  - {note}')